# Sessione 7 · Progettazione del progetto

**Modulo 4 · La tua prima app di IA** · 2 sessioni · progetto reale · dall'idea al prodotto

> Arriviamo al progetto finale: costruiremo un **chatbot di Domande Frequenti (FAQ) per l'assistenza clienti**. Il cliente scrive il suo dubbio e l'assistente risponde basandosi su un PDF di FAQ. Oggi progettiamo la soluzione e montiamo il primo prototipo; nella Sessione 8 lo raffineremo e lo pubblicheremo.

**Al termine della sessione saprai:**
- Definire un problema reale e progettarne la soluzione
- Capire l'architettura **dati → modello → interfaccia**
- Leggere un PDF e convertirlo in contesto per un LLM
- Creare una prima interfaccia web con **Streamlit**

*Risultato della sessione: hai un prototipo funzionante del chatbot.*

> 📎 **Ti serve il file `faq_horta_verde.pdf`** (te lo consegno insieme al notebook). Mettilo nella stessa cartella. Su Google Colab, caricalo con il pannello file a sinistra.

## 1. Il problema: cosa risolviamo e per chi?

**Il caso.** Un negozio online (il fittizio *Horta Verde*) riceve ogni giorno le stesse domande: tempi di spedizione, resi, modalità di pagamento... Il suo team perde ore a rispondere alle stesse cose e, fuori dall'orario d'ufficio, il cliente resta senza risposta.

**La soluzione.** Un assistente disponibile 24/7 che risponde a quei dubbi all'istante, **usando come unica fonte l'informazione ufficiale dell'azienda** (il suo documento di FAQ). Non inventa nulla e non sostituisce il team umano: lo libera dal lavoro ripetitivo.

**Perché un LLM è adatto qui.** Le domande arrivano in linguaggio naturale e in mille forme diverse ("quando arriva?", "tempi di consegna?", "per quando lo ricevo?"). Un sistema a regole non le coprirebbe tutte; un LLM capisce l'intenzione e risponde con naturalezza. È esattamente ciò che abbiamo visto nel Modulo 3.

> **Principio di progettazione:** un buon primo progetto risolve **un problema piccolo e concreto**. Meglio un chatbot che padroneggia le FAQ di un negozio che un "assistente che sa tutto".

## 2. L'architettura: dati → modello → interfaccia

Ogni app di IA, per quanto complessa possa sembrare, segue questo schema a tre blocchi:

```
   DATI                 MODELLO                 INTERFACCIA
┌──────────┐        ┌──────────────┐        ┌──────────────┐
│ PDF      │  ───>  │ LLM (Claude, │  ───>  │ Web di chat  │
│ delle FAQ│        │ Llama...)    │        │ (Streamlit)  │
└──────────┘        └──────────────┘        └──────────────┘
   Il testo         Riceve le FAQ            Il cliente
   di riferimento   + la domanda e           scrive e legge
                     redige la risposta      la risposta
```

**Come "diamo" le FAQ al modello?** Qui prendiamo una decisione di progettazione importante. Dato che il documento di FAQ è **breve**, usiamo la tecnica più semplice ed efficace: **mettere tutto il testo delle FAQ dentro il *system prompt***. Il modello riceve a ogni richiesta le regole + le FAQ complete + la domanda del cliente. Questo si chiama, informalmente, *"context-stuffing"* (riempire il contesto).

> Quando il documento è enorme (centinaia di pagine), inserirlo per intero non entra nella finestra di contesto e si usa un'altra tecnica chiamata **RAG** (ricerca + generazione). La lasciamo per i prossimi passi (Sessione 8). Per delle FAQ, il *context-stuffing* è l'opzione corretta e semplice.

## 3. Passo 1 · Leggere il PDF delle FAQ

Iniziamo dai **dati**. Usiamo la libreria `pypdf` per estrarre il testo dal PDF. Esegui le celle (installa prima se serve).

In [ ]:
!pip install pypdf anthropic openai ollama streamlit -q
print("Pronto ✅")

In [ ]:
from pypdf import PdfReader

lector = PdfReader("faq_horta_verde.pdf")
texto_faq = ""
for pagina in lector.pages:
    texto_faq += pagina.extract_text() + "\n"

print("Pagine lette:", len(lector.pages))
print("Caratteri estratti:", len(texto_faq))
print("\n--- Prime righe ---")
print(texto_faq[:400])

## 4. Passo 2 · Costruire il prompt

Ora prepariamo le istruzioni dell'assistente (il **system prompt**) applicando quanto imparato nella Sessione 5: gli diamo un **ruolo**, delle **regole chiare** e la **fonte di informazione** (le FAQ). La regola più importante: che risponda **solo** con ciò che è nelle FAQ e che, se non lo sa, lo dica.

In [ ]:
system = f"""Sei l'assistente dell'assistenza clienti di Horta Verde, un negozio online.
Rispondi in italiano, in modo cordiale e breve.

Regole:
- Usa SOLO le informazioni delle FAQ qui sotto.
- Se la risposta non c'è, dillo e invita a scrivere a hola@hortaverde.example. Non inventare.

=== FAQ ===
{texto_faq}
=== FINE ==="""

print("System prompt preparato.")
print("Lunghezza (caratteri):", len(system))

## 5. Passo 3 · Interrogare l'LLM

Testiamo la logica prima di incapsularla in una web. Riutilizziamo quanto visto nel Modulo 3. Inserisci la tua chiave e lancia una domanda di prova.

> Mostriamo l'esempio con **Claude**, ma l'app finale permetterà di scegliere tra i tre fornitori (Claude, OpenRouter gratuito e Ollama locale), esattamente come nella Sessione 6.

In [ ]:
import os
from getpass import getpass
os.environ["ANTHROPIC_API_KEY"] = getpass("Chiave Anthropic: ")

In [ ]:
import anthropic

def responder_faq(pregunta):
    cliente = anthropic.Anthropic()
    r = cliente.messages.create(
        model="claude-haiku-4-5",
        max_tokens=400,
        system=system,
        messages=[{"role": "user", "content": pregunta}],
    )
    return r.content[0].text

# Una domanda che È nelle FAQ
print(responder_faq("Quanto costa la spedizione?"))
print("\n----\n")
# Una domanda che NON c'è -> dovrebbe riconoscere di non saperlo
print(responder_faq("Avete un negozio fisico a Madrid?"))

## 6. L'interfaccia: cos'è Streamlit?

**Streamlit** è una libreria che trasforma uno script Python in una **web interattiva**, senza sapere nulla di sviluppo web. Tu scrivi Python e Streamlit mette i pulsanti, le caselle di testo e la chat.

C'è una differenza chiave rispetto a quanto fatto finora: un'app Streamlit **non si esegue dentro il notebook**, ma come un programma a parte. Il flusso è:
1. Scrivi il codice in un file `.py`
2. Lo lanci dal terminale con `streamlit run mi_app.py`
3. Si apre nel browser

Dal notebook useremo il comando magico `%%writefile` per **salvare** il codice in un file, e poi lo eseguiremo nel terminale.

In [ ]:
%%writefile hola_streamlit.py
import streamlit as st

st.title("Ciao, Streamlit!")
nombre = st.text_input("Come ti chiami?")
if nombre:
    st.write(f"Piacere, {nombre} 👋")

La cella precedente **non mostra nulla**: ha solo salvato il file `hola_streamlit.py`. Per vederlo in azione, apri un terminale nella stessa cartella ed esegui:

```bash
streamlit run hola_streamlit.py
```

> Su **Google Colab**, eseguire Streamlit richiede un passo extra (un tunnel). Il modo più comodo di seguire questa parte è sul tuo **computer**: installa Python, o usa l'editor che preferisci, e lancia i comandi `streamlit run` nel suo terminale. Se lavori su Colab, dimmelo e ti passo la ricetta del tunnel.

## 7. Il tuo primo prototipo

Mettiamo insieme i tre pezzi (leggere il PDF + prompt + LLM) dentro un'app Streamlit minima. Questa versione fa l'essenziale: una casella per la domanda, un pulsante e la risposta. Ancora senza memoria della conversazione: la aggiungiamo nella Sessione 8.

La cella seguente **salva** il prototipo in `app_v1.py`.

In [ ]:
%%writefile app_v1.py
# -*- coding: utf-8 -*-
"""Prototipo v1 del chatbot di FAQ (Sessione 7).
Versione minima: una domanda, una risposta. Senza cronologia per ora.

Eseguire con:  streamlit run app_v1.py
"""
import os
import streamlit as st
from pypdf import PdfReader

st.title("Assistente FAQ (prototipo)")

# 1. Leggere il PDF delle FAQ
lector = PdfReader("faq_horta_verde.pdf")
texto_faq = ""
for pagina in lector.pages:
    texto_faq += pagina.extract_text() + "\n"

# 2. Chiedere la chiave di Claude (solo per il prototipo; poi la miglioreremo)
clave = st.text_input("La tua chiave Anthropic", type="password")

# 3. Casella per la domanda del cliente
pregunta = st.text_input("Come posso aiutarti?")

# 4. Al clic del pulsante, interroghiamo l'LLM
if st.button("Chiedi") and pregunta and clave:
    import anthropic
    cliente = anthropic.Anthropic(api_key=clave)
    system = (
        "Sei l'assistente dell'assistenza clienti di Horta Verde. "
        "Rispondi in italiano, breve e cordiale, usando SOLO questa informazione:\n\n"
        + texto_faq
    )
    respuesta = cliente.messages.create(
        model="claude-haiku-4-5",
        max_tokens=400,
        system=system,
        messages=[{"role": "user", "content": pregunta}],
    )
    st.write(respuesta.content[0].text)


Ora eseguilo nel tuo terminale:

```bash
streamlit run app_v1.py
```

Si aprirà nel browser. Inserisci la tua chiave, scrivi *"quanto ci mette la spedizione?"* e premi **Chiedi**. Hai appena usato la tua prima app di IA! 🎉

👉 **Provalo con 5 domande diverse**, alcune presenti nelle FAQ e altre no. Annota come si comporta: lo miglioreremo insieme.

## 8. Riepilogo della sessione

- Un buon progetto risolve **un problema piccolo e concreto**: qui, le FAQ di un negozio
- Ogni app di IA è **dati → modello → interfaccia**
- Per un documento breve, la tecnica più semplice è **mettere le FAQ nel system prompt** (*context-stuffing*)
- **Streamlit** trasforma uno script Python in una web; si lancia con `streamlit run`
- Hai già un **prototipo funzionante** (`app_v1.py`)

**Per la prossima sessione:** lo trasformiamo in un prodotto: interfaccia di chat con memoria, logica più robusta, pubblicazione online e demo finale.

👉 **Porta i tuoi appunti** dei test del prototipo. Li useremo per decidere cosa rifinire.